In [1]:
import pandas as pd 
import nltk
nltk.download('universal_tagset')
from nltk.tag import pos_tag
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from googletrans import Translator
import time
import re
import gensim.downloader as api
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt')
nltk.download('stopwords')
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
import multiprocessing
from deep_translator import GoogleTranslator

[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
c:\Users\filip\anaconda3\envs\Y2_BlockA\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\filip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Select language of the original sentences

In [2]:
selected_language = 'english'  # Specify the language of the input data here

### Load in the data from csv file and specify column with the sentences

In [3]:
sentences = pd.read_csv(r'..\Data\CSV\recording_sentences_AssemlyAI_10_9.csv')
sentence_column_name = 'Sentence'

In [4]:
def translate_to_english(text, source_language):
    # Skip translation if source language is already English
    if source_language.lower() in ['en', 'english']:
        return text
    
    try:
        translator = GoogleTranslator(source=source_language, target='en')
        result = translator.translate(text)
        return result if result else text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

# Apply to DataFrame
sentences['translated_text'] = sentences.apply(
    lambda row: translate_to_english(row[sentence_column_name], selected_language), 
    axis=1
)

## Sentence related features

Part-of-Speech (POS) Tagging:
Extract POS tags for each sentence in your transcription using an NLP library such as SpaCy or NLTK.
Create a new column in your dataset named ‘POS_Tags’ and store the POS tags for each sentence.

Sentiment Analysis:
Perform sentiment analysis on each sentence using a library like TextBlob or VADER or another one that you find suiting for your data.
Add a column ‘Sentiment_Score’ to your dataset with the sentiment polarity score for each sentence.

Pretrained Word Embeddings:
Use pretrained word embeddings (e.g., Word2Vec, GloVe) to convert the sentences in your dataset into vector representations.
Describe what happens with words that are not present in the pre-trained word embedding model.

### POS tags

In [5]:
def POS_tagging(sentences):
    sentences['POS_tags'] = sentences['translated_text'].apply(lambda x: pos_tag(word_tokenize(x), tagset='universal'))
    return sentences
sentences = POS_tagging(sentences)

### Sentiment

In [42]:
def sentiment_analysis(sentences):
    sentences['Sentiment'] = sentences['translated_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
    return sentences
sentences = sentiment_analysis(sentences)

### TF-IDF

In [7]:
def tfidf_vectorization(sentences):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(sentences['translated_text'])

    # Convert sparse matrix to dense and then to list of arrays
    tfidf_dense = tfidf_matrix.toarray()

    # Add the TF-IDF vectors as a new column
    sentences['TF-IDF'] = [row for row in tfidf_dense]
    return sentences
sentences = tfidf_vectorization(sentences)


### Pretrained Word Embeddings

In [44]:
# Load pretrained Word2Vec model
model = api.load("word2vec-google-news-300")

def word2vec_embedding(model, sentences):
    vectors = []
    all_missing_words = []
    
    for sentence in sentences['translated_text']:
        words = sentence.lower().split()
        word_vectors = []
        missing_words = []
        
        for word in words:
            try:
                word_vectors.append(model[word])
            except KeyError:
                missing_words.append(word)
        
        if word_vectors:
            # Average all word vectors in the sentence
            sentence_vector = np.mean(word_vectors, axis=0)
        else:
            # If no words found, return zero vector
            sentence_vector = np.zeros(300)
        
        vectors.append(sentence_vector)
        all_missing_words.extend(missing_words)

    return vectors, all_missing_words, word_vectors

vectors, all_missing_words, word_vectors = word2vec_embedding(model, sentences)
# Add the vectors as a new column
sentences['word2vec_embedding'] = vectors

#### Describe what happens with words that are not present in the pre-trained word embedding model.

When word is not present in the pre-trained word embedding model, the vector of that word is all zeroes. Meanwhile if a word is present, it is appropriately processed

In [22]:
print("Missing word example (vector):", vectors[0])

print("Non-missing word example (vector):", vectors[1])

Missing word example (vector): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Non-missing word example (vector): [ 1.15331011e-02  5.

### Custom Word Embeddings

The data used for custom embeddings are Yelp Restaurant reviews. Since our show is related to kitchen, restaurant, cooking, we figured we would need to look for something related to that. At first we two things came to our minds. First was cooking forums where people shared recipes and second one was restaurant reviews. Considering that we want to focus on classifying emotion with this project, we believe that the Yelp reviews provide us with more emotions present than cooking forums with cooking recipes and discussions. 

In [45]:
reviews = pd.read_csv(r'..\Data\CSV\Yelp Restaurant Reviews.csv')

In [46]:
def yelp_custom_embeddings(reviews):
    reviews = reviews.drop(columns=['Yelp URL','Rating','Date'])
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', x))
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'\S+@\S+', '', x))
    reviews['Review Text'] = reviews['Review Text'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))
    raw_corpus = reviews['Review Text'].tolist()
    return raw_corpus
raw_corpus = yelp_custom_embeddings(reviews)

In [47]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs, email addresses, special characters
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def tokenize_corpus(corpus):
    sentences = []
    
    for document in corpus:
        # Clean the document
        clean_doc = preprocess_text(document)
        
        # Split into sentences
        doc_sentences = sent_tokenize(clean_doc)
        
        for sentence in doc_sentences:
            # Tokenize words and remove very short sentences
            words = word_tokenize(sentence)
            if len(words) >= 3:  # Keep sentences with at least 3 words
                sentences.append(words)
    
    return sentences

# Preprocess your corpus
processed_sentences = tokenize_corpus(raw_corpus)

In [48]:
# Analyze corpus characteristics
total_sentences = len(processed_sentences)
total_words = sum(len(sentence) for sentence in processed_sentences)
vocab_size = len(set(word for sentence in processed_sentences for word in sentence))

print(f"Corpus Statistics:")
print(f"Total sentences: {total_sentences:,}")
print(f"Total words: {total_words:,}")
print(f"Vocabulary size: {vocab_size:,}")
print(f"Average sentence length: {total_words/total_sentences:.1f}")

# Show sample sentences
print(f"\nSample processed sentences:")
for i in range(3):
    print(f"Sentence {i+1}: {' '.join(processed_sentences[i])}")

Corpus Statistics:
Total sentences: 19,892
Total words: 1,798,747
Vocabulary size: 27,186
Average sentence length: 90.4

Sample processed sentences:
Sentence 1: all i can say is they have very good ice cream i would for sure recommend their cookies and creme ice cream it is very good
Sentence 2: nice little local place for ice creammy favorite is their pumpkin shake fall season special my sweetness tolerance is low their large size ice cream usually seems too sweet after having ice cream for a while but love their pina colada so refreshing their banana split is good too
Sentence 3: a delicious treat on a hot day staff was very friendly and helpful gave us a sample and let us order a little earlier than open


#### Motivate the choice of parameters used

**Core Model Settings:**

vector_size = 300: Each word gets represented as a 300-number vector. This is like giving each word a "fingerprint" with 300 measurements - enough detail to capture meaning without being too complex.

window = 5: When learning about a word, the model looks at 5 words before and 5 words after it. This helps capture how words relate to their immediate neighbors.

min_count = 5: Ignores any word that appears fewer than 5 times in your text. This filters out typos, rare words, and noise that might confuse the model.

**Training Approach:**

sg = 1: Uses "Skip-gram" method, which tries to predict surrounding words from a center word (like seeing "king" and guessing "royal" might be nearby). The alternative (sg=0) is CBOW, which does the reverse.

epochs = 30: The model will go through your entire text 30 times to learn the patterns. More epochs = better learning, but takes longer.

**Performance Tuning:**

workers = multiprocessing.cpu_count(): Uses all your computer's CPU cores to train faster in parallel.

alpha = 0.025: The learning rate - how big steps the model takes when adjusting. 0.025 is a standard starting point that usually works well.

negative = 20: A training trick that makes the model learn faster by using "negative examples" - teaching it what words DON'T go together.

In [49]:
def train_word2vec_model(processed_sentences):
    
    # Set hyperparameters based on corpus analysis
    
    vector_size = 300      # 300 dimensions - good balance between expressiveness and efficiency
    window = 5            # 5 words context window - captures local semantic relationships
    min_count = 5         # Ignore words appearing less than 5 times - removes noise
    workers = multiprocessing.cpu_count()  # Use all CPU cores
    sg = 1                # Skip-gram model (sg=1) vs CBOW (sg=0)
    epochs = 30           # Number of training iterations
    alpha = 0.025         # Initial learning rate
    negative = 20         # Negative sampling - speeds up training
    
    # Train the Word2Vec model

    model = Word2Vec(
        sentences=processed_sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg,
        epochs=epochs,
        alpha=alpha,
        negative=negative
        )
    return model, vector_size
word2vec_model, vector_size = train_word2vec_model(processed_sentences)

In [50]:
def create_sentence_embeddings(sentences, model, vector_size, text_column='translated_text', embedding_column='custom_word2vec_embedding'):
    """
    Create sentence embeddings using Word2Vec model by averaging word vectors.
    
    Parameters:
    -----------
    sentences : pandas.DataFrame
        DataFrame containing the sentences to embed
    model : gensim.models.Word2Vec
        Trained Word2Vec model
    vector_size : int
        Size of the word vectors (dimensionality)
    text_column : str, default 'Sentence'
        Name of the column containing the text to embed
    embedding_column : str, default 'custom_word2vec_embedding'
        Name of the column to store the embeddings
    
    Returns:
    --------
    pandas.DataFrame
        Original DataFrame with added embedding column
    list
        List of words that were not found in the model vocabulary
    """
    
    custom_vectors = []
    custom_missing_words = []
    
    for sentence in sentences[text_column]:
        words = sentence.lower().split()
        word_vectors = []
        missing_words = []
        
        for word in words:
            try:
                word_vectors.append(model[word])
            except KeyError:
                missing_words.append(word)
        
        if word_vectors:
            sentence_vector = np.mean(word_vectors, axis=0)
        else:
            sentence_vector = np.zeros(vector_size)
        
        custom_vectors.append(sentence_vector)
        custom_missing_words.extend(missing_words)
    
    sentences[embedding_column] = custom_vectors
    
    return sentences, custom_missing_words


sentences_with_embeddings, missing_words,  = create_sentence_embeddings(
     sentences=sentences, 
     model=model, 
     vector_size=vector_size)

In [51]:
print(f"Missing words: {len(set(missing_words))} unique words")
print(f"Total missing occurrences: {len(missing_words)}")

Missing words: 719 unique words
Total missing occurrences: 2034


### Additional feature

In [52]:
def create_bert_embeddings(df, text_column, pooling_method='cls', max_length=512, batch_size=8):
    """
    Create BERT embeddings for sentences in a DataFrame column
    
    Parameters:
    df: DataFrame containing the text data
    text_column: Name of the column containing sentences
    pooling_method: 'cls' for CLS token or 'mean' for mean pooling
    max_length: Maximum sequence length for BERT
    batch_size: Batch size for processing
    
    Returns:
    df: DataFrame with new column 'bert_embedding' added
    """
    from transformers import BertTokenizer, BertModel
    import torch
    import numpy as np
    
    # Load pre-trained BERT model and tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')
    
    # Set model to evaluation mode
    model.eval()
    
    def get_bert_embeddings(sentences, method='cls'):
        embeddings = []
        
        for i in range(0, len(sentences), batch_size):
            batch_sentences = sentences[i:i+batch_size]
            
            # Tokenize the batch
            encoded = tokenizer(
                batch_sentences,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            
            # Get embeddings without computing gradients
            with torch.no_grad():
                outputs = model(**encoded)
                
                if method == 'cls':
                    # Use CLS token embedding
                    batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                else:  # mean pooling
                    # Mean pooling with attention mask
                    attention_mask = encoded['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
                    sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, 1)
                    sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                    batch_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
                
                embeddings.extend(batch_embeddings)
        
        return np.array(embeddings)
    
    # Convert sentences to BERT embeddings
    sentences_list = df[text_column].tolist()
    bert_embeddings = get_bert_embeddings(sentences_list, method=pooling_method)
    
    # Add BERT embeddings as a new column to dataframe
    df['bert_embedding'] = [emb for emb in bert_embeddings]
    
    # Print verification results
    print(f"BERT embedding shape: {bert_embeddings.shape}")
    print(f"Each sentence embedding shape: {df['bert_embedding'].iloc[0].shape}")
    print(f"DataFrame shape: {df.shape}")
    print(f"Total sentences processed: {len(sentences_list)}")
    print(f"Embedding dimensions: {bert_embeddings.shape[1]}")
    print(f"Pooling method used: {pooling_method}")
    
    return df

In [53]:
sentences = create_bert_embeddings(sentences, 'translated_text')

c:\Users\filip\anaconda3\envs\Y2_BlockA\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BERT embedding shape: (1038, 768)
Each sentence embedding shape: (768,)
DataFrame shape: (1038, 8)
Total sentences processed: 1038
Embedding dimensions: 768
Pooling method used: cls


In [54]:
sentences

,Sentence,translated_text,POS_tags,Sentiment,TF-IDF,word2vec_embedding,custom_word2vec_embedding,bert_embedding
0,"Laverne, California.","Laverne, California.","[(Laverne, NOUN), (,, .), (California, NOUN), ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.41221687, -0.12478843, -0.40210357, -0.183..."
1,"Just 30 miles from Los Angeles, this suburb is...","Just 30 miles from Los Angeles, this suburb is...","[(Just, ADV), (30, NUM), (miles, NOUN), (from,...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2950870124125...","[0.011533101, 0.05309367, 0.13485718, 0.086883...","[0.011533101, 0.05309367, 0.13485718, 0.086883...","[0.38464928, -0.21587497, 0.121124044, -0.2438..."
2,"And in 2010, it was bought by the Leyva family.","And in 2010, it was bought by the Leyva family.","[(And, CONJ), (in, ADP), (2010, NUM), (,, .), ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.43236119930387173, 0.0,...","[0.052022297, 0.017326036, 0.07470703, 0.05369...","[0.052022297, 0.017326036, 0.07470703, 0.05369...","[-0.21337116, 0.06332427, 0.16847925, -0.54158..."
3,Would you like a booth?,Would you like a booth?,"[(Would, VERB), (you, PRON), (like, ADP), (a, ...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.13232422, 0.09358724, 0.095199585, 0.214843...","[0.13232422, 0.09358724, 0.095199585, 0.214843...","[0.20054688, 0.024506789, -0.22534221, -0.1112..."
4,I started working here 10 years ago.,I started working here 10 years ago.,"[(I, PRON), (started, VERB), (working, VERB), ...",0.0,"[0.42853089523033905, 0.0, 0.0, 0.0, 0.0, 0.0,...","[-0.0703125, 0.10585938, 0.014611816, 0.108007...","[-0.0703125, 0.10585938, 0.014611816, 0.108007...","[0.1559361, 0.18722928, 0.055368703, -0.640785..."
...,...,...,...,...,...,...,...,...
1033,But so is her relationship with her family.,But so is her relationship with her family.,"[(But, CONJ), (so, ADV), (is, VERB), (her, PRO...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.028438022, -0.012276785, 0.000919887, 0.027...","[0.028438022, -0.012276785, 0.000919887, 0.027...","[-0.23598924, 0.06054914, -0.49875048, -0.2591..."
1034,I owe Chef Ramsay like everything.,I owe Chef Ramsay like everything.,"[(I, PRON), (owe, VERB), (Chef, NOUN), (Ramsay...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.057617188, -0.029846191, 0.112781525, 0.10...","[-0.057617188, -0.029846191, 0.112781525, 0.10...","[0.0076824455, 0.58142674, -0.0631088, -0.4402..."
1035,He has molded me into the business owner I nee...,He has molded me into the business owner I nee...,"[(He, PRON), (has, VERB), (molded, VERB), (me,...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007220459, 0.005822754, -0.009115601, -0.00...","[0.007220459, 0.005822754, -0.009115601, -0.00...","[-0.18150908, 0.30177516, -0.08130963, -0.2715..."
1036,"When he comes back, he's gonna be like, you've...","When he comes back, he's gonna be like, you've...","[(When, ADV), (he, PRON), (comes, VERB), (back...",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.108947754, 0.008560181, 0.058883667, 0.1559...","[0.108947754, 0.008560181, 0.058883667, 0.1559...","[-0.2600104, 0.06395439, 0.28019783, -0.124955..."


In [57]:
# Save the first 10 rows with semicolon as delimiter
sentences.head(10).to_csv(r'..\Data\CSV\NLP_features.csv', index=False, sep= ';')